# ROGII — General Stack v2 (training + CV, no submission)

Physics signals + spatial-imputer features + alignment offsets → LGBM GroupKFold.
Baselines: const 15.9 · PF ~10.8 · stack v1 10.62. Goal: CV ≤ 9.

In [ ]:
import os

In [ ]:
"""
Particle-filter geosteering tracker (numpy, vectorised over particles).

State per particle: (pos, rate) where pos = TVT + Z  ("structural" position that
is ~constant when geology is flat), rate = d(pos)/dMD. At each eval step we
propagate with momentum, convert to TVT = pos - Z, score against the typewell
GR(TVT) curve via a Gaussian likelihood, resample, and take the weighted-mean TVT.

Predictions are made only for the eval rows (TVT_input is NaN). A seed ensemble
is combined with weights ∝ exp(loglik/scale). This is the core signal that gets
geosteering solutions to single-digit RMSE.
"""
import numpy as np

# tuned defaults (close to strong public solutions)
MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5


def _run_pf(md, z, gr, tw_tvt, tw_gr, last_tvt, last_Z, ir, gs,
            n_particles=500, seed=0):
    rng = np.random.default_rng(seed)
    N = n_particles
    ls = last_tvt + last_Z
    pos = ls + 2.0 * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N) / N
    lo, hi = tw_tvt[0] - 100, tw_tvt[-1] + 100
    out = np.empty(len(md)); loglik = 0.0
    prev_md = md[0] - 1.0
    for i in range(len(md)):
        dm = max(md[i] - prev_md, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dm + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - z[i], lo, hi)
        pos = tvt_p + z[i]
        if np.isfinite(gr[i]):
            eg = np.interp(tvt_p, tw_tvt, tw_gr)
            d = (gr[i] - eg) / gs
            lk = np.maximum(np.exp(-0.5 * np.minimum(d * d, 600.0)), 1e-300)
            avg = float((w * lk).sum()); loglik += np.log(max(avg, 1e-300))
            w = w * lk; s = w.sum(); w = w / s if s > 0 else np.ones(N) / N
        neff = 1.0 / (w * w).sum()
        if neff < RESAMP * N:
            cum = np.cumsum(w); u0 = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos = pos[idx] + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w = np.ones(N) / N
        out[i] = float(np.dot(w, pos - z[i]))
        prev_md = md[i]
    return out, loglik


def pf_predict(hw, tw, n_particles=500, n_seeds=48, scale=5.0):
    """Return a full-length TVT array (known rows kept, eval rows predicted)."""
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)

    kn = hw[hw["TVT_input"].notna()]
    ev = hw[hw["TVT_input"].isna()]
    out = hw["TVT_input"].to_numpy(float).copy()
    if len(ev) == 0:
        return out

    last = kn.iloc[-1]
    last_tvt = float(last["TVT_input"]); last_Z = float(last["Z"])
    # GR noise sigma from known section
    tw_at_k = np.interp(kn["TVT_input"].to_numpy(), tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn["GR"].fillna(0).to_numpy() - tw_at_k), 10., 60.))
    # initial structural rate from last 30 known points
    tail = kn.tail(30)
    dt = np.diff(tail["TVT_input"].to_numpy()); dz = np.diff(tail["Z"].to_numpy())
    dmv = np.diff(tail["MD"].to_numpy()); m = dmv > 0
    ir = float(np.median((dt + dz)[m] / dmv[m])) if m.sum() >= 3 else 0.0

    gr_all = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    md = ev["MD"].to_numpy(float); z = ev["Z"].to_numpy(float)
    gr = gr_all[ev.index.to_numpy()]
    # prepend last known so momentum starts from the right place
    md = np.concatenate([[float(last["MD"])], md])
    z = np.concatenate([[last_Z], z])
    gr = np.concatenate([[np.nan], gr])

    preds = np.empty((n_seeds, len(md))); liks = np.empty(n_seeds)
    for s in range(n_seeds):
        preds[s], liks[s] = _run_pf(md, z, gr, tw_tvt, tw_gr, last_tvt, last_Z,
                                    ir, gs, n_particles, seed=s)
    liks -= liks.max()
    wts = np.exp(liks / scale); wts /= wts.sum()
    est = (wts[:, None] * preds).sum(0)[1:]        # drop the prepended known row
    out[ev.index.to_numpy()] = est
    return out


In [ ]:
"""Beam-search geosteering tracker (numba JIT). Complements the particle filter:
finds a low-cost monotone-ish path through the typewell GR(TVT) grid that matches
the horizontal GR signature. Ensembled over several stiffness configs."""
import numpy as np
from numba import njit
from scipy.signal import savgol_filter

# (beam_size, move_cost, err_scale, savgol_radius)
BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2), (10, 8.0, 64.0, 2), (8, 35.0, 220.0, 1),
    (10, 14.0, 90.0, 5), (20, 4.0, 36.0, 3), (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
]


@njit(cache=False)
def _beam(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS * 6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.0; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt:
                    continue
                tot = cost + (gv - tw_gr[ni]) ** 2 / es + mc * (d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni:
                        fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]:
                        cC[fnd] = tot; cP[fnd] = bi
                elif nc < MAX:
                    cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i + 1, nc):
                if cC[j] < cC[mi]:
                    mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]
                cC[i], cC[mi] = cC[mi], cC[i]
                cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]:
            best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n - 1, -1, -1):
        path[s] = hI[s, b]; b = hP[s, b]
    return path


def beam_predict(hw, tw):
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    out = hw["TVT_input"].to_numpy(float).copy()
    if len(ev) == 0:
        return out
    last_tvt = float(kn.iloc[-1]["TVT_input"])
    gr_all = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    hgr = gr_all[ev.index.to_numpy()]
    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))
    paths = []
    for BS, mc, es, r in BEAM_CONFIGS:
        if r > 0 and len(hgr) > max(3, 2 * r + 1):
            win = min(2 * r + 1, len(hgr) if len(hgr) % 2 == 1 else len(hgr) - 1)
            sgr = savgol_filter(hgr, win, min(2, win - 1))
        else:
            sgr = hgr.copy()
        paths.append(tw_tvt[_beam(sgr.astype(np.float64), tw_gr, si, BS, float(mc), float(es))])
    out[ev.index.to_numpy()] = np.stack(paths, 0).mean(0)
    return out


In [ ]:
import os, glob, time, math
import numpy as np, pandas as pd
from scipy.spatial import cKDTree
from joblib import Parallel, delayed

def find_root():
    for r in ["/kaggle/input/rogii-wellbore-geology-prediction",
              "/kaggle/input/competitions/rogii-wellbore-geology-prediction","data"]+sorted(glob.glob("/kaggle/input/*")):
        if glob.glob(os.path.join(r,"train","*__horizontal_well.csv")): return r
    hits=glob.glob("/kaggle/input/**/train/*__horizontal_well.csv",recursive=True)
    if hits: return os.path.dirname(os.path.dirname(hits[0]))
    raise FileNotFoundError("data")
DATA=find_root(); print("DATA:",DATA,flush=True)
FORMS=["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
ANCH_OFF=[-40,-20,-10,-5,0,5,10,20,40]

def ps_index(h): return int(h["TVT_input"].notna().sum())

# ---------- spatial reference models (fit on all train wells; leave-self-out at query) ----------
def build_plane_index():
    rows=[]
    for f in sorted(glob.glob(f"{DATA}/train/*__horizontal_well.csv")):
        w=os.path.basename(f).split("__")[0]
        try: df=pd.read_csv(f,usecols=["X","Y"]+FORMS).dropna()
        except Exception: continue
        if not len(df): continue
        r={"wid":w,"x":float(df["X"].median()),"y":float(df["Y"].median())}
        for c in FORMS: r[c]=float(df[c].median())
        rows.append(r)
    idx=pd.DataFrame(rows)
    xy=idx[["x","y"]].to_numpy(); scl=np.where(xy.std(0)<1e-3,1.,xy.std(0))
    return idx, cKDTree(xy/scl), scl

PIDX,PTREE,PSCL=build_plane_index()
PXA=PIDX["x"].to_numpy(); PYA=PIDX["y"].to_numpy()
PFA=PIDX[FORMS].to_numpy(np.float64); PWID={w:i for i,w in enumerate(PIDX["wid"])}

def impute_forms(xy_q, self_wid, k=10):
    q=xy_q/PSCL; nf=min(k+5,len(PIDX))
    dist,ii=PTREE.query(q,k=nf,workers=-1)
    if self_wid in PWID: dist=np.where(ii==PWID[self_wid],np.inf,dist)
    order=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
    dk=np.take_along_axis(dist,order,1); ik=np.take_along_axis(ii,order,1)
    vk=np.isfinite(dk); wt=np.where(vk,1./(dk+1e-3),0.)
    xn=PXA[ik]; yn=PYA[ik]; fn=PFA[ik]; wx=wt*xn; wy=wt*yn
    n=len(q); A=np.zeros((n,3,3))
    A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
    A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
    A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=wt.sum(1)
    for d in range(3): A[:,d,d]+=1e-9
    rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(wt[:,:,None]*fn).sum(1)],1)
    try: coef=np.linalg.solve(A,rhs)
    except Exception: coef=np.stack([np.linalg.pinv(A[r])@rhs[r] for r in range(n)])
    pred=xy_q[:,0][:,None]*coef[:,0,:]+xy_q[:,1][:,None]*coef[:,1,:]+coef[:,2,:]
    dmin=np.where(vk,dk,np.inf).min(1)
    return pred, dmin

def multiscale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
    out=[]
    for hw in hws:
        win=2*hw+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0:
            out.append((np.full(nh,ktvt[-1] if len(ktvt) else 0.,np.float32),np.zeros(nh,np.float32))); continue
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().to_numpy(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().to_numpy(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32)
        if not len(sts):
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]]
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw,mode="edge")
        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]]
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))
    return out

def robust_slope(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<2 or np.std(x[m])<1e-6: return 0.
    return float(np.polyfit(x[m],y[m],1)[0])

def seg_b(ktvt,kz,fcol):
    bv=ktvt+kz-fcol; n=len(bv)
    b_full=float(np.median(bv))
    b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
    return b_full,b_late

def build_well(path,is_train,self_excl):
    wid=os.path.basename(path).split("__")[0]; base=os.path.dirname(path)
    try:
        h=pd.read_csv(path); tw=pd.read_csv(f"{base}/{wid}__typewell.csv")
    except Exception: return None
    ps=ps_index(h)
    if ps<60 or ps>=len(h)-3: return None
    if is_train and "TVT" not in h.columns: return None
    tw_s=tw.sort_values("TVT"); tw_tvt=tw_s["TVT"].to_numpy(float)
    tw_gr=tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)
    if len(tw_tvt)<3: return None
    tps=float(h["TVT_input"].iloc[ps-1])
    md=h["MD"].to_numpy(float); x=h["X"].to_numpy(float); y=h["Y"].to_numpy(float); z=h["Z"].to_numpy(float)
    gr=h["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    ev=np.arange(ps,len(h)); nev=len(ev)
    ktvt=h["TVT_input"].to_numpy(float)[:ps]; kz=z[:ps]; kmd=md[:ps]; kgr=gr[:ps]

    # physics candidate paths: two PF ensembles (disagreement = uncertainty) + beam
    pf_a=pf_predict(h,tw,n_particles=300,n_seeds=16,scale=12.0)
    pf_b=pf_predict(h,tw,n_particles=300,n_seeds=16,scale=3.0)
    bm=beam_predict(h,tw)
    ncc=multiscale_ncc(kgr,ktvt,gr[ps:])

    # spatial features
    form_ev,dmin_ev=impute_forms(np.column_stack([x[ev],y[ev]]), wid if self_excl else None)
    form_kn,_=impute_forms(np.column_stack([x[:ps],y[:ps]]), wid if self_excl else None)
    sp={}
    cands=[]
    for fi,fn in enumerate(FORMS):
        bf,bl=seg_b(ktvt,kz,form_kn[:,fi])
        c_full=(-z[ev]+form_ev[:,fi]+bf); c_late=(-z[ev]+form_ev[:,fi]+bl)
        sp[f"spF_{fn}"]=(c_full-tps).astype(np.float32)
        sp[f"spL_{fn}"]=(c_late-tps).astype(np.float32)
        kn_rmse=float(np.sqrt(np.mean((ktvt-(-kz+form_kn[:,fi]+bf))**2)))
        sp[f"spRm_{fn}"]=np.float32(kn_rmse)
        cands.append(c_full)
    cst=np.stack(cands,1)
    sp["sp_mean_d"]=(cst.mean(1)-tps).astype(np.float32)
    sp["sp_std"]=cst.std(1).astype(np.float32)
    sp["sp_dmin"]=dmin_ev.astype(np.float32)

    tw_at=lambda t: np.interp(t,tw_tvt,tw_gr)
    # typewell "supertype" fingerprint (57 unique typewells shared across wells):
    # wells sharing a typewell must stay in the same CV fold (no typewell leakage)
    twh=int(abs(hash((round(float(tw_tvt[0]),1),round(float(tw_tvt[-1]),1),
                      round(float(np.nansum(tw_gr[:200])),0),len(tw_tvt))))%10**9)
    gr_s=pd.Series(gr)
    f={"well":wid,"id":[f"{wid}_{i}" for i in ev],"last_known_tvt":tps,
       "supertype":twh,
       "pf_d":(pf_a[ev]-tps).astype(np.float32),
       "pf3_d":(pf_b[ev]-tps).astype(np.float32),
       "pf_dis":(pf_a[ev]-pf_b[ev]).astype(np.float32),
       "beam_d":(bm[ev]-tps).astype(np.float32),
       "pf_vs_beam":(pf_a[ev]-bm[ev]).astype(np.float32),
       "ncc8_d":np.clip(ncc[0][0]-tps,-80,80),"ncc8_s":ncc[0][1],
       "ncc15_d":np.clip(ncc[1][0]-tps,-80,80),"ncc15_s":ncc[1][1],
       "ncc25_d":np.clip(ncc[2][0]-tps,-80,80),"ncc25_s":ncc[2][1],
       "d_md":(md[ev]-md[ps-1]).astype(np.float32),
       "d_z":(z[ev]-z[ps-1]).astype(np.float32),
       "d_xy":np.hypot(x[ev]-x[ps-1],y[ev]-y[ps-1]).astype(np.float32),
       "dz_dmd":(np.gradient(z)/(np.gradient(md)+1e-9))[ev].astype(np.float32),
       "gr":gr[ev].astype(np.float32),
       "gr_m21":gr_s.rolling(21,center=True,min_periods=1).mean().to_numpy()[ev].astype(np.float32),
       "gr_s21":gr_s.rolling(21,center=True,min_periods=1).std().fillna(0).to_numpy()[ev].astype(np.float32),
       "frac":(np.arange(nev)/max(nev-1,1)).astype(np.float32),
       "slp_all":np.float32(robust_slope(kmd,ktvt)),
       "slp_50":np.float32(robust_slope(kmd[-50:],ktvt[-50:])),
       "ktvt_rng":np.float32(np.ptp(ktvt)),"ktvt_std":np.float32(ktvt.std()),
       "pfx_gr_rmse":np.float32(np.sqrt(np.nanmean((kgr-tw_at(ktvt))**2))),
      }
    for o in ANCH_OFF:
        f[f"tda{o}"]=(gr[ev]-tw_at(tps+o)).astype(np.float32)
        f[f"tdpf{o}"]=(gr[ev]-tw_at(pf_a[ev]+o)).astype(np.float32)
    f.update(sp)
    if is_train:
        f["target"]=(h["TVT"].to_numpy(float)[ev]-tps).astype(np.float32)
    return pd.DataFrame(f)

t0=time.time()
paths=sorted(glob.glob(f"{DATA}/train/*__horizontal_well.csv"))
res=Parallel(n_jobs=4)(delayed(build_well)(p,True,True) for p in paths)
train=pd.concat([r for r in res if r is not None],ignore_index=True)
print("train",train.shape,"wells",train["well"].nunique(),"%.0fs"%(time.time()-t0),flush=True)
train.to_parquet("train_feats.parquet") if hasattr(pd.DataFrame,"to_parquet") else train.to_pickle("train_feats.pkl")

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
import joblib

DROP={"well","id","target","last_known_tvt","supertype"}
FEATS=[c for c in train.columns if c not in DROP]
print("features",len(FEATS))
print("unique typewell supertypes:",train["supertype"].nunique())
X=train[FEATS].to_numpy(np.float32); yv=train["target"].to_numpy(np.float32)
g=train["supertype"].to_numpy()      # fold by typewell supertype (57) — no typewell leakage
P=dict(objective="regression",n_estimators=1500,learning_rate=0.02,num_leaves=127,
       min_child_samples=100,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,
       reg_lambda=5.0,reg_alpha=1.0,verbose=-1,n_jobs=-1)
gkf=GroupKFold(n_splits=5); oof=np.zeros(len(yv)); models=[]
for k,(tr,va) in enumerate(gkf.split(X,yv,g)):
    m=lgb.LGBMRegressor(**P).fit(X[tr],yv[tr])
    oof[va]=m.predict(X[va]); models.append(m); print("fold",k,"done",flush=True)

def rr(t,p): return float(np.sqrt(np.mean((t-p)**2)))
print("const  pooled %.3f"%rr(yv,0))
print("pf     pooled %.3f"%rr(yv,train["pf_d"].to_numpy()))
print("stack2 pooled %.3f"%rr(yv,oof))
best=(1e9,1.0,100)
for sh in [1.0,0.95,0.9,0.85,0.8]:
    for cl in [40,60,100]:
        r=rr(yv,np.clip(oof*sh,-cl,cl))
        if r<best[0]: best=(r,sh,cl)
print("stack2 pp pooled %.3f (shrink=%.2f clip=%d)"%best)
imp=pd.Series(models[0].feature_importances_,index=FEATS).sort_values(ascending=False)
print("top20 features:"); print(imp.head(20).to_string())
np.save("oof.npy",oof); joblib.dump({"models":models,"feats":FEATS,"pp":best[1:]},"stack_v2.joblib")
print("DONE",flush=True)